In [ ]:
# Google Drive link for trained weights
# https://drive.google.com/file/d/1O00CmSy6WorkplYfU7shvjLni6mRGEmP/view?usp=sharing

In [ ]:
!pip install gdown
!gdown --fuzzy "https://drive.google.com/file/d/1O00CmSy6WorkplYfU7shvjLni6mRGEmP/view?usp=sharing"
#downloads the trained best.pt

In [ ]:
!nvidia-smi

In [ ]:
!pip install sahi ultralytics #yolo, sahi in this package

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
seq_path = "/content/drive/MyDrive/dataset/images/uav0000086_00000_v"
print(len(os.listdir(seq_path)))

In [ ]:
!pip install boxmot

In [ ]:
!pip install bytetracker

In [ ]:
!pip install lapx

In [ ]:
from ultralytics.trackers.byte_tracker import BYTETracker
print("ByteTrack imported successfully")

In [ ]:
!pip install supervision

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import supervision as sv
import time

def estimate_camera_motion(prev_gray, curr_gray): #estimates how drone cam moved b/w two frames
    #detects good corners in previous frame to track
    prev_pts = cv2.goodFeaturesToTrack(prev_gray, maxCorners=200, qualityLevel=0.01, minDistance=30)
    if prev_pts is None:
        return np.eye(2, 3, dtype=np.float32)  # identity — no correction
    # track those corners into current frame using optical flow
    curr_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_pts, None)

    # keep only successfully tracked points
    prev_pts = prev_pts[status.flatten() == 1]
    curr_pts = curr_pts[status.flatten() == 1]

    if len(prev_pts) < 4:
        return np.eye(2, 3, dtype=np.float32)

    # estimate affine transform — this is the camera motion matrix
    matrix, _ = cv2.estimateAffinePartial2D(prev_pts, curr_pts)
    if matrix is None:
        return np.eye(2, 3, dtype=np.float32)
    return matrix

def apply_cmc_to_detections(dets, matrix): #purpose is to shift detection boxes acc to camera movement
    if len(dets) == 0:
        return dets
    # apply affine transform to all detection center points
    centers = np.array([[(d[0]+d[2])/2, (d[1]+d[3])/2] for d in dets])
    centers_h = np.hstack([centers, np.ones((len(centers), 1))])
    corrected = (matrix @ centers_h.T).T
    # reconstruct boxes from corrected centers
    result = dets.copy()
    w = dets[:, 2] - dets[:, 0]
    h = dets[:, 3] - dets[:, 1]
    result[:, 0] = corrected[:, 0] - w/2
    result[:, 1] = corrected[:, 1] - h/2
    result[:, 2] = corrected[:, 0] + w/2
    result[:, 3] = corrected[:, 1] + h/2
    return result

#path
MODEL_PATH  = "/content/best.pt"
FRAMES_DIR  = "/content/drive/MyDrive/dataset/images/uav0000086_00000_v"
OUTPUT_PATH = "/content/output.mp4"

#loading model into SAHI wrapper
#SAHI wraps Yolov8 so it can run sliced inference
# confidence threshold is 0.3 , we keeping it lower because drone person are small
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=MODEL_PATH,
    confidence_threshold=0.3,
    device="cuda"
)

#byteTrack args
tracker = sv.ByteTrack() #assign unique ids to detected peeople

#video-writing->creates the output.mp4
frames = sorted(Path(FRAMES_DIR).glob("*.jpg"))
sample = cv2.imread(str(frames[0]))
h, w = sample.shape[:2]
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, 10, (w, h))

#list of center points-stored as trajectory storage
trajectories = {} #stores the mobvement history

#Tracking FPS
fps_list = []

print(f"Processing {len(frames)} frames...")

prev_gray = None

for i, frame_path in enumerate(frames):
    frame = cv2.imread(str(frame_path))
    t_start = time.time()
    curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    #we convert to greyscale since needed for optical flow

    #sahi sliced inference, slice size kept 640 na zyada kam na zyada bada, and no rescaling each tile, bcz yolo ka image size is 640X640 pixel, overlap is 0.2 to ensure person near edges appear fully in atleast one tile
    result = get_sliced_prediction(
        frame,
        detection_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
    )

    # convert SAHI output to supervision Detections format
    xyxy = []
    confs = []
    for pred in result.object_prediction_list:
        xyxy.append([pred.bbox.minx, pred.bbox.miny, pred.bbox.maxx, pred.bbox.maxy])
        confs.append(pred.score.value)

    if xyxy:
        detections = sv.Detections(
            xyxy=np.array(xyxy, dtype=np.float32),
            confidence=np.array(confs, dtype=np.float32),
            class_id=np.zeros(len(xyxy), dtype=int)
        )
    else:
        detections = sv.Detections.empty()

    # CMC — apply camera motion compensation before tracker update
    if prev_gray is not None and len(xyxy) > 0:
        matrix = estimate_camera_motion(prev_gray, curr_gray)
        corrected_xyxy = apply_cmc_to_detections(np.array(xyxy, dtype=np.float32), matrix)
        detections = sv.Detections(
            xyxy=corrected_xyxy,
            confidence=np.array(confs, dtype=np.float32),
            class_id=np.zeros(len(xyxy), dtype=int)
        )

    prev_gray = curr_gray

    # ByteTrack update — returns [x1, y1, x2, y2, track_id, conf, class, ...]
    tracked = tracker.update_with_detections(detections)

    t_end = time.time()
    fps_list.append(1.0 / (t_end - t_start))

    #drawing boxes,IDs,trajectory tails
    for j in range(len(tracked)):
        x1, y1, x2, y2 = map(int, tracked.xyxy[j])
        tid = int(tracked.tracker_id[j])
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

        #storing the trajectory
        if tid not in trajectories:
            trajectories[tid] = []
        trajectories[tid].append((cx, cy))

        #drawing b.box and ID
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"ID {tid}", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        #trajectory tail of last 30 points
        tail = trajectories[tid][-30:]
        for k in range(1, len(tail)):
            cv2.line(frame, tail[k-1], tail[k], (0, 0, 255), 1)

    out.write(frame)
    if i % 50 == 0:
        print(f"Frame {i}/{len(frames)} | FPS: {fps_list[-1]:.2f}")

out.release()
avg_fps = np.mean(fps_list)
print(f"\nDone. Output: {OUTPUT_PATH}")
print(f"Average FPS: {avg_fps:.2f}")
print(f"Min FPS: {min(fps_list):.2f} | Max FPS: {max(fps_list):.2f}")

In [ ]:
from google.colab import files
files.download('/content/output.mp4')

In [ ]:
!pip show bytetracker

In [ ]:
import boxmot
print(dir(boxmot))


In [ ]:
import cv2
import numpy as np
from pathlib import Path
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import supervision as sv
import time

def estimate_camera_motion(prev_gray, curr_gray):
    #detects good corners in previous frame to track
    prev_pts = cv2.goodFeaturesToTrack(prev_gray, maxCorners=200, qualityLevel=0.01, minDistance=30)
    if prev_pts is None:
        return np.eye(2, 3, dtype=np.float32)  # identity — no correction
    # track those corners into current frame using optical flow
    curr_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_pts, None)

    # keep only successfully tracked points
    prev_pts = prev_pts[status.flatten() == 1]
    curr_pts = curr_pts[status.flatten() == 1]

    if len(prev_pts) < 4:
        return np.eye(2, 3, dtype=np.float32)

    # estimate affine transform — this is the camera motion matrix
    matrix, _ = cv2.estimateAffinePartial2D(prev_pts, curr_pts)
    if matrix is None:
        return np.eye(2, 3, dtype=np.float32)
    return matrix

def apply_cmc_to_detections(dets, matrix):
    if len(dets) == 0:
        return dets
    # apply affine transform to all detection center points
    centers = np.array([[(d[0]+d[2])/2, (d[1]+d[3])/2] for d in dets])
    centers_h = np.hstack([centers, np.ones((len(centers), 1))])
    corrected = (matrix @ centers_h.T).T
    # reconstruct boxes from corrected centers
    result = dets.copy()
    w = dets[:, 2] - dets[:, 0]
    h = dets[:, 3] - dets[:, 1]
    result[:, 0] = corrected[:, 0] - w/2
    result[:, 1] = corrected[:, 1] - h/2
    result[:, 2] = corrected[:, 0] + w/2
    result[:, 3] = corrected[:, 1] + h/2
    return result

#path
MODEL_PATH  = "/content/best.pt"
FRAMES_DIR  = "/content/drive/MyDrive/dataset/images/uav0000305_00000_v"
OUTPUT_PATH = "/content/output.mp4"

#loading model into SAHI wrapper
#SAHI wraps Yolov8 so it can run sliced inference
# confidence threshold is 0.3 , we keeping it lower because drone person are small
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=MODEL_PATH,
    confidence_threshold=0.3,
    device="cuda"
)

#byteTrack args
tracker = sv.ByteTrack()

#video-writing
frames = sorted(Path(FRAMES_DIR).glob("*.jpg"))
sample = cv2.imread(str(frames[0]))
h, w = sample.shape[:2]
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, 10, (w, h))

#list of center points-stored as trajectory storage
trajectories = {}

#Tracking FPS
fps_list = []

print(f"Processing {len(frames)} frames...")

prev_gray = None

for i, frame_path in enumerate(frames):
    frame = cv2.imread(str(frame_path))
    t_start = time.time()
    curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    #sahi sliced inference, slice size kept 640 na zyada kam na zyada bada, and no rescaling each tile, bcz yolo ka image size is 640X640 pixel, overlap is 0.2 to ensure person near edges appear fully in atleast one tile
    result = get_sliced_prediction(
        frame,
        detection_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
    )

    # convert SAHI output to supervision Detections format
    xyxy = []
    confs = []
    for pred in result.object_prediction_list:
        xyxy.append([pred.bbox.minx, pred.bbox.miny, pred.bbox.maxx, pred.bbox.maxy])
        confs.append(pred.score.value)

    if xyxy:
        detections = sv.Detections(
            xyxy=np.array(xyxy, dtype=np.float32),
            confidence=np.array(confs, dtype=np.float32),
            class_id=np.zeros(len(xyxy), dtype=int)
        )
    else:
        detections = sv.Detections.empty()

    # CMC — apply camera motion compensation before tracker update
    if prev_gray is not None and len(xyxy) > 0:
        matrix = estimate_camera_motion(prev_gray, curr_gray)
        corrected_xyxy = apply_cmc_to_detections(np.array(xyxy, dtype=np.float32), matrix)
        detections = sv.Detections(
            xyxy=corrected_xyxy,
            confidence=np.array(confs, dtype=np.float32),
            class_id=np.zeros(len(xyxy), dtype=int)
        )

    prev_gray = curr_gray

    # ByteTrack update — returns [x1, y1, x2, y2, track_id, conf, class, ...]
    tracked = tracker.update_with_detections(detections)

    t_end = time.time()
    fps_list.append(1.0 / (t_end - t_start))

    #drawing boxes,IDs,trajectory tails
    for j in range(len(tracked)):
        x1, y1, x2, y2 = map(int, tracked.xyxy[j])
        tid = int(tracked.tracker_id[j])
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

        #storing the trajectory
        if tid not in trajectories:
            trajectories[tid] = []
        trajectories[tid].append((cx, cy))

        #drawing b.box and ID
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"ID {tid}", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        #trajectory tail of last 30 points
        tail = trajectories[tid][-30:]
        for k in range(1, len(tail)):
            cv2.line(frame, tail[k-1], tail[k], (0, 0, 255), 1)

    out.write(frame)
    if i % 50 == 0:
        print(f"Frame {i}/{len(frames)} | FPS: {fps_list[-1]:.2f}")

out.release()
avg_fps = np.mean(fps_list)
print(f"\nDone. Output: {OUTPUT_PATH}")
print(f"Average FPS: {avg_fps:.2f}")
print(f"Min FPS: {min(fps_list):.2f} | Max FPS: {max(fps_list):.2f}")

In [ ]:
import supervision as sv
print([x for x in dir(sv) if 'motion' in x.lower() or 'cmc' in x.lower() or 'camera' in x.lower()])